# UC Question Sprint



In [ ]:
import pandas as pd

bay = pd.read_csv("bay_area_modeling_table.csv")
eth = pd.read_csv("uc_admissions_summary_by_ethnicity.csv")
disc = pd.read_csv("uc_freshman_admission_by_discipline.csv")
transfer = pd.read_csv("uc_transfer_admission_by_major.csv")

for name, df in [("bay", bay), ("eth", eth), ("disc", disc), ("transfer", transfer)]:
    print(name, df.shape)

## Q1. Fall 2025: average number of UC campuses per applicant
Uses `uc_admissions_summary_by_ethnicity.csv`. Each campus row = applicants to that campus; the `Systemwide` row = distinct (unduplicated) applicants. Sum of per-campus applicants ÷ systemwide distinct applicants = average campuses applied to.

In [ ]:
d25 = eth[(eth.fall_term == 2025) & (eth.count_type == 'App')]

for level in ['freshman', 'transfer']:
    sub = d25[d25.entrant_level == level]
    per_campus_sum = sub[sub.campus != 'Systemwide']['n'].sum()
    systemwide = sub[sub.campus == 'Systemwide']['n'].sum()
    avg = per_campus_sum / systemwide
    print(f"{level}: {avg:.2f} campuses/applicant  (per-campus sum={per_campus_sum}, systemwide={systemwide})")

# combined freshman + transfer
per_campus_all = d25[d25.campus != 'Systemwide']['n'].sum()
systemwide_all = d25[d25.campus == 'Systemwide']['n'].sum()
print(f"combined: {per_campus_all/systemwide_all:.2f} campuses/applicant")

## Q2. Fall 2025 admit rate at UCLA for Bay Area public high school applicants
Uses `bay_area_modeling_table.csv`, campus = `Los Angeles`, `school_type` containing "Public".

In [ ]:
ucla25 = bay[(bay.campus == 'Los Angeles') & (bay.fall_term == 2025)]
pub = ucla25[ucla25.school_type.astype(str).str.contains('Public', na=False)]

tot_app = pub['applicants'].sum()
tot_adm = pub['admits'].sum()
print(f"Bay Area public HS -> UCLA, fall 2025: {tot_adm:.0f} admits / {tot_app:.0f} applicants = {tot_adm/tot_app:.4f}")

# for comparison: UCLA overall (all applicants, all origins), from the discipline file
ucla_all = disc[(disc.campus == 'Los Angeles') & (disc.fall_term == 2025) & (disc.broad_discipline == 'All disciplines')]
print(ucla_all[['applicants', 'admits', 'admit_rate']])

## Q3. Fall 2025: which UC campus's Computer Science admit rate lags its overall admit rate the most?
Uses `uc_freshman_admission_by_discipline.csv`, comparing each campus's `Computer Science` row to its `All disciplines` row.

In [ ]:
d25disc = disc[disc.fall_term == 2025]

cs = d25disc[d25disc.broad_discipline == 'Computer Science'][['campus','applicants','admits','admit_rate']]
cs = cs.rename(columns={'applicants':'cs_app','admits':'cs_adm','admit_rate':'cs_rate'})

allc = d25disc[d25disc.broad_discipline == 'All disciplines'][['campus','applicants','admits','admit_rate']]
allc = allc.rename(columns={'applicants':'all_app','admits':'all_adm','admit_rate':'all_rate'})

m = cs.merge(allc, on='campus')
m['gap_pts'] = (m['all_rate'] - m['cs_rate']) * 100
m = m.sort_values('gap_pts', ascending=False)
print(m.to_string(index=False))
print("\nBiggest CS penalty:", m.iloc[0]['campus'], f"({m.iloc[0]['gap_pts']:.1f} pts)")

## Q4. Fall 2025: IQR of admit GPA for Berkeley Computer Science
Uses `uc_freshman_admission_by_discipline.csv`, `admit_gpa_p25` / `admit_gpa_p75` columns.

In [ ]:
row = disc[(disc.fall_term == 2025) & (disc.campus == 'Berkeley') & (disc.broad_discipline == 'Computer Science')]
p25 = row['admit_gpa_p25'].values[0]
p75 = row['admit_gpa_p75'].values[0]
print(f"p25={p25}, p75={p75}, IQR={p75 - p25:.2f}")

## Q5. Fall 2025: at how many of the 9 UC campuses was the White freshman admit rate higher than Hispanic/Latino(a)?
Uses `uc_admissions_summary_by_ethnicity.csv`, per-campus rows only (excludes `Systemwide`).

In [ ]:
d = eth[(eth.fall_term == 2025) & (eth.entrant_level == 'freshman') & (eth.campus != 'Systemwide') &
        (eth.ethnicity.isin(['White', 'Hispanic/Latino(a)']))]

results = []
for campus in d.campus.unique():
    row = {'campus': campus}
    for e in ['White', 'Hispanic/Latino(a)']:
        app = d[(d.campus == campus) & (d.ethnicity == e) & (d.count_type == 'App')]['n'].values[0]
        adm = d[(d.campus == campus) & (d.ethnicity == e) & (d.count_type == 'Adm')]['n'].values[0]
        row[e] = adm / app
    results.append(row)

res = pd.DataFrame(results)
res['white_higher'] = res['White'] > res['Hispanic/Latino(a)']
print(res.to_string(index=False))
print(f"\nCampuses where White admit rate > Hispanic/Latino(a): {res['white_higher'].sum()} of {len(res)}")

## Q6. Systemwide, fall 2025: White vs. Hispanic/Latino(a) freshman admit rate — which is higher?
Same source, `campus == 'Systemwide'` this time.

In [ ]:
d_sys = eth[(eth.fall_term == 2025) & (eth.entrant_level == 'freshman') & (eth.campus == 'Systemwide') &
            (eth.ethnicity.isin(['White', 'Hispanic/Latino(a)']))]
piv = d_sys.pivot_table(index='ethnicity', columns='count_type', values='n')
piv['admit_rate'] = piv['Adm'] / piv['App']
print(piv)
print("\nHigher systemwide admit rate:", piv['admit_rate'].idxmax())

## Q7 (bonus). Class of 2023: share of Bay Area HS graduates who enrolled at a California Community College within 12 months
Uses `bay_area_modeling_table.csv`, `campus == 'Universitywide'`, `fall_term == 2023`, `enrolled_ccc` / `hs_completers`.

In [ ]:
uw23 = bay[(bay.campus == 'Universitywide') & (bay.fall_term == 2023)]
tot_completers = uw23['hs_completers'].sum()
tot_ccc = uw23['enrolled_ccc'].sum()
print(f"CCC enrollment share, class of 2023: {tot_ccc:.0f} / {tot_completers:.0f} = {tot_ccc/tot_completers:.4f}")

## Q8 (bonus). Mission San Jose HS, fall 2023: share of a-g completers who applied to at least one UC
`applicants` (Universitywide) ÷ `ag_completers`.

In [ ]:
row = bay[(bay.high_school.str.contains('MISSION SAN JOSE', case=False, na=False)) &
          (bay.campus == 'Universitywide') & (bay.fall_term == 2023)]
share = row['applicants'].values[0] / row['ag_completers'].values[0]
print(f"applicants={row['applicants'].values[0]:.0f}, ag_completers={row['ag_completers'].values[0]:.0f}, share={share:.4f}")

## Q9 (bonus, use with caution). Fall 2025: distinct Bay Area public high schools sending >=1 UC applicant
**Caveat:** `bay_area_modeling_table.csv` covers only the 9-county Bay Area, not all of California, and `school_type` has some missing labels for schools that are plausibly public charters (join gap in the source file). The cell below reports a few different cuts rather than one single number

In [ ]:
uw25 = bay[(bay.campus == 'Universitywide') & (bay.fall_term == 2025) & (bay.applicants > 0)]

print("All school types, distinct high_school names:", uw25['high_school'].nunique())

pub_labeled = uw25[uw25.school_type.astype(str).str.contains('Public', na=False)]
print("Labeled 'Public' school_type only:", pub_labeled['high_school'].nunique())

narrow = uw25[uw25.school_type == 'High Schools (Public)']
print("Narrowly 'High Schools (Public)' only:", narrow['high_school'].nunique())

## Q10 (bonus). Fall 2022-2025: which school MOST outperforms its expected UC Berkeley admit rate?
Controls for a-g completion rate, FRPM % (poverty), applicant GPA, and applicant pool size via linear regression; ranks schools by residual (actual − predicted admit rate).

**Caveat:** in-sample R^2 is low (~0.15) — treat this as directional, not precise. `applicant_gpa` also looks like it may be a merge artifact (identical value across years for a given school), so weigh that skepticism into any conclusion.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

berk = bay[(bay.campus == 'Berkeley') & (bay.fall_term.between(2022, 2025))].copy()
feats = ['ag_completion_rate', 'frpm_pct', 'applicant_gpa', 'applicants']
target = 'admit_rate'

m = berk.dropna(subset=feats + [target]).copy()
X = m[feats]
y = m[target]

scaler = StandardScaler()
Xs = scaler.fit_transform(X)
lr = LinearRegression().fit(Xs, y)
print("R2 (in-sample):", lr.score(Xs, y))

m['predicted'] = lr.predict(Xs)
m['residual_pts'] = (m['admit_rate'] - m['predicted']) * 100

rank = m.groupby('high_school')['residual_pts'].mean().sort_values(ascending=False)
print(rank.head(10))